In [253]:
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoLatestQuoteRequest
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime
import pandas as pd
import time
import random

In [254]:
# no keys required.
client = CryptoHistoricalDataClient()


In [255]:
request_params = CryptoLatestQuoteRequest(symbol_or_symbols="ETH/USD")
latest_quote = client.get_crypto_latest_quote(request_params)

In [256]:
latest_quote

{'ETH/USD': {   'ask_exchange': None,
     'ask_price': 2959.8,
     'ask_size': 23.1734,
     'bid_exchange': None,
     'bid_price': 2954.168,
     'bid_size': 23.126,
     'conditions': None,
     'symbol': 'ETH/USD',
     'tape': None,
     'timestamp': datetime.datetime(2025, 11, 26, 0, 33, 18, 505596, tzinfo=TzInfo(0))}}

## Tokens Alpaca API supports:
#### USD pairs:
AAVE, AVAX, BAT, BCH, BTC, CRV, DOGE, DOT, ETH,  
GRT, LINK, LTC, PEPE, SHIB, SKY, SOL, SUSHI,  
TRUMP, UNI, USDC, USDG, USDT, XRP, XTZ, YFI

In [ ]:
# stable coins: USDC, USDG, USDT
# bluechip: BTC, ETH

# 19 tradable tokens
tokens = [
  'AAVE', 'AVAX', 'BAT', 'BCH', 'CRV', 'DOGE', 'DOT', 'GRT', 'LINK', 'LTC',
  'PEPE', 'SHIB', 'SKY', 'SOL', 'SUSHI', 'UNI', 'XRP', 'XTZ', 'YFI',
  'BTC', 'ETH'  # Keep these in for now
]

for i,v in enumerate(tokens):
  t = tokens[i].strip()
  tokens[i] = f'{t}/USD'

In [258]:
len(tokens)

21

## Fetch the tokens
(Can take a long time... 15 min!)

In [259]:
batch_size = 1
all_dfs = []

for i in range(0, len(tokens), batch_size):
  batch = tokens[i : i + batch_size]
  print(f'{i} {batch}', end='\t')

  request_params = CryptoBarsRequest(
    symbol_or_symbols=batch,
    timeframe=TimeFrame.Hour,
    start=datetime(2020, 1, 1),
    end=datetime(2025, 11, 26),
  )
  bars = client.get_crypto_bars(request_params)
  all_dfs.append(bars.df.reset_index())

  time.sleep(3 + (9 - 3) * random.random())

  print('Done.')

df = pd.concat(all_dfs, ignore_index=True)

0 ['AAVE/USD']	Done.
1 ['AVAX/USD']	Done.
2 ['BAT/USD']	Done.
3 ['BCH/USD']	Done.
4 ['CRV/USD']	Done.
5 ['DOGE/USD']	Done.
6 ['DOT/USD']	Done.
7 ['GRT/USD']	Done.
8 ['LINK/USD']	Done.
9 ['LTC/USD']	Done.
10 ['PEPE/USD']	Done.
11 ['SHIB/USD']	Done.
12 ['SKY/USD']	Done.
13 ['SOL/USD']	Done.
14 ['SUSHI/USD']	Done.
15 ['UNI/USD']	Done.
16 ['XRP/USD']	Done.
17 ['XTZ/USD']	Done.
18 ['YFI/USD']	Done.
19 ['BTC/USD']	Done.
20 ['ETH/USD']	Done.


In [279]:
len(df['symbol'].unique())

KeyError: 'symbol'

In [261]:
df.head()

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,AAVE/USD,2021-07-15 13:00:00+00:00,269.83,284.00,268.00,274.16,489.7873,297.0,272.678919
1,AAVE/USD,2021-07-15 14:00:00+00:00,273.96,276.53,272.06,273.88,74.7332,36.0,273.894978
2,AAVE/USD,2021-07-15 15:00:00+00:00,275.00,275.56,272.11,272.11,128.1683,38.0,274.725797
3,AAVE/USD,2021-07-15 16:00:00+00:00,272.00,273.10,267.69,269.64,78.5813,47.0,270.677157
4,AAVE/USD,2021-07-15 17:00:00+00:00,269.18,269.18,265.86,266.97,32.9152,23.0,266.922659


In [262]:
df['timestamp'].min()

Timestamp('2021-01-01 06:00:00+0000', tz='UTC')

In [263]:
df['timestamp'].max()

Timestamp('2025-11-26 00:00:00+0000', tz='UTC')

In [283]:
samples = len(df)
if str(request_params.timeframe) == str(TimeFrame.Minute):
  samples //= 60  # hourly
  samples //= 24  # daily
elif str(request_params.timeframe) == str(TimeFrame.Hour):
  samples //= 24  # daily

print(f'Current samples:\t\t{len(df):,}')
print(f'Projected dailyy samples:\t{samples:,}')
print(f'Projected hourly samples:\t{samples * 24:,}')
print(f'Projected minute samples:\t{samples * 24 * 60:,}')

Current samples:		668,030
Projected dailyy samples:	27,834
Projected hourly samples:	668,016
Projected minute samples:	40,080,960


In [265]:
df.rename({'symbol':'token', 'timestamp':'date'}, axis=1, inplace=True)
df = df.drop(['trade_count', 'vwap'], axis=1)

In [266]:
df.head()

,token,date,open,high,low,close,volume
0,AAVE/USD,2021-07-15 13:00:00+00:00,269.83,284.00,268.00,274.16,489.7873
1,AAVE/USD,2021-07-15 14:00:00+00:00,273.96,276.53,272.06,273.88,74.7332
2,AAVE/USD,2021-07-15 15:00:00+00:00,275.00,275.56,272.11,272.11,128.1683
3,AAVE/USD,2021-07-15 16:00:00+00:00,272.00,273.10,267.69,269.64,78.5813
4,AAVE/USD,2021-07-15 17:00:00+00:00,269.18,269.18,265.86,266.97,32.9152


In [ ]:
df.groupby('token').count().sort_values(by='date', ascending=False)

,date,open,high,low,close,volume
token,,,,,,
BTC/USD,42942,42942,42942,42942,42942,42942
ETH/USD,42940,42940,42940,42940,42940,42940
LTC/USD,42938,42938,42938,42938,42938,42938
DOGE/USD,42932,42932,42932,42932,42932,42932
LINK/USD,42930,42930,42930,42930,42930,42930
UNI/USD,42872,42872,42872,42872,42872,42872
BCH/USD,42822,42822,42822,42822,42822,42822
BAT/USD,42381,42381,42381,42381,42381,42381
GRT/USD,38225,38225,38225,38225,38225,38225


In [284]:
df.to_csv('../data/crypto_data_new.csv', index=False)